In [1]:
library(tidyverse)
library(data.table)
library(plotly)
library(glmnet)

Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”
Warning message:
“package ‘tibble’ was built under R version 4.4.3”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
“package ‘data.table’ was built under R version 4.4.3”

Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:dplyr’:

    between, first, last


The

In [33]:
process_df <- function(df) {
    df <- fread(df)
    df$GO_ID <- as.factor(df$GO_ID)
    df <- df %>% pivot_longer(cols = c("div_d_gamb", "div_d_colu", "div_d_arab"), names_to = "Species", values_to = "Polymorphism")
    df$Species <- as.factor(df$Species)
    df <- df[!is.na(df$Polymorphism) & !is.infinite(df$Polymorphism) & df$Polymorphism > 0,]
    df$Level <- as.factor(df$Level)
    return(df)
}

lambda_cv <- function(gomatrix, df, alpha=1) {
    #df_matrix <- model.matrix(~GO_ID, data=df)

    divergence_vector <- df$fake_stat
    lambda_seq <- 10^seq(3, -3, by=-0.1)

    cv_output <- cv.glmnet(gomatrix, divergence_vector, alpha=alpha, lambda=lambda_seq, nfolds=10)
    best_lambda <- cv_output$lambda.min
    return(best_lambda)
}

lasso_regress <- function(gomatrix, df, alpha, lambda){
    #df_matrix <- model.matrix(~GO_ID, data=df)
    divergence_vector <- df$fake_stat

    lasso_model <- glmnet(gomatrix, divergence_vector, alpha=alpha, lambda=lambda)
    coefficients <- as.data.frame(as.matrix(coef(lasso_model)))
    colnames(coefficients) <- "Coefficient"
    coefficients <- rownames_to_column(coefficients, var="GO_ID")
    non_zero_coefficients <- coefficients[coefficients$Coefficient != 0,]
    return(non_zero_coefficients)
}

In [ ]:
statpath='/home/crehmann/vectorcomp/out/gamb_colu_arab_gene_stats_wide_GO_IDs.txt' 

gene_stats <- fread(statpath)
colnames(gene_stats)

[1] "Gene"                      "Chromosome"               
 [3] "Transcript"                "gene_pi_gamb"             
 [5] "gene_pi_colu"              "gene_pi_arab"             
 [7] "aa_pi_gamb"                "aa_pi_colu"               
 [9] "aa_pi_arab"                "cds_pi_gamb"              
[11] "cds_pi_colu"               "cds_pi_arab"              
[13] "ss_pi_gamb"                "ss_pi_colu"               
[15] "ss_pi_arab"                "ns_pi_gamb"               
[17] "ns_pi_colu"                "ns_pi_arab"               
[19] "gene_theta_gamb"           "gene_theta_colu"          
[21] "gene_theta_arab"           "aa_theta_gamb"            
[23] "aa_theta_colu"             "aa_theta_arab"            
[25] "cds_theta_gamb"            "cds_theta_colu"           
[27] "cds_theta_arab"            "ss_theta_gamb"            
[29] "ss_theta_colu"             "ss_theta_arab"            
[31] "ns_theta_gamb"             "ns_theta_colu"            
[33] "ns_theta_arab"             "gene_d_gamb"              
[35] "gene_d_colu"               "gene_d_arab"              
[37] "aa_d_gamb"                 "aa_d_colu"                
[39] "aa_d_arab"                 "cds_d_gamb"               
[41] "cds_d_colu"                "cds_d_arab"               
[43] "ss_d_gamb"                 "ss_d_colu"                
[45] "ss_d_arab"                 "ns_d_gamb"                
[47] "ns_d_colu"                 "ns_d_arab"                
[49] "div_d_gamb"                "div_d_colu"               
[51] "div_d_arab"                "source_id"                
[53] "Computed GO Component IDs" "Computed GO Components_1" 
[55] "Computed GO Components_2"  "Computed GO Components_3" 
[57] "Computed GO Components_4"  "Computed GO Components_5" 
[59] "Computed GO Components_6"  "Computed GO Components_7" 
[61] "Computed GO Components_8"  "Computed GO Components_9" 
[63] "Computed GO Components_10" "Computed GO Function IDs" 
[65] "Computed GO Functions_1"   "Computed GO Functions_2"  
[67] "Computed GO Functions_3"   "Computed GO Functions_4"  
[69] "Computed GO Functions_5"   "Computed GO Functions_6"  
[71] "Computed GO Functions_7"   "Computed GO Functions_8"  
[73] "Computed GO Functions_9"   "Computed GO Functions_10" 
[75] "Computed GO Functions_11"  "Computed GO Functions_12" 
[77] "Computed GO Process IDs"   "Computed GO Processes_1"  
[79] "Computed GO Processes_2"   "Computed GO Processes_3"  
[81] "Computed GO Processes_4"   "Computed GO Processes_5"  
[83] "Computed GO Processes_6"   "Computed GO Processes_7"  
[85] "Computed GO Processes_8"   "Computed GO Processes_9"  
[87] "Computed GO Processes_10"  "Computed GO Processes_11" 
[89] "Curated GO Component IDs"  "Curated GO Components"    
[91] "Curated GO Function IDs"   "Curated GO Functions"     
[93] "Curated GO Process IDs"    "Curated GO Processes"     
[95] "EC numbers"                "EC numbers from OrthoMCL"

In [27]:
gene_stats <- gene_stats %>% separate_longer_delim(cols = c("Curated GO Component IDs"), delim=';')

In [29]:
gene_stats$fake_stat <- 0
gene_stats[gene_stats$`Curated GO Component IDs` == "GO:0005737",]$fake_stat <- 10

In [30]:
compmat <- fread('out/gamb_colu_arab_Component_edge_matrix.txt')
compmat <- t(as.matrix(compmat))
compmat <- as.data.frame(compmat)
names(compmat) <- as.character(unlist(compmat[1,]))
compmat <- compmat[-1,]
compmat <- tibble::rownames_to_column(compmat, var="Gene")

In [31]:
mdf <- merge(gene_stats, compmat)

In [35]:
L <- lambda_cv(as.matrix(select(mdf, contains("GO:"))), mdf, alpha=1)
print(L)
coefs <- lasso_regress(as.matrix(select(mdf, contains("GO:"))), mdf, alpha=1, lambda=L)
coefs %>% arrange(desc(Coefficient))

[1] 0.01258925


GO_ID,Coefficient
<chr>,<dbl>
GO:0005737;GO:0016528,4.22933897
GO:0005575;GO:0110165,0.76314600
GO:0005871;GO:0005872,0.64992645
GO:0098798;GO:0017087,0.44579283
GO:0043232;GO:0005814,0.40894312
(Intercept),0.30812557
GO:0030120;GO:0030126,0.30797353
GO:0036056;GO:0005917,0.30048823
GO:0031410;GO:0099503,0.27021113
